<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/mobilidade/Transito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mortes no trânsito — taxa por 100 mil habitantes

Este notebook é o motor de coleta e publicação da série `mortes-transito-anual`.

- **Fonte:** Ministério da Saúde — Sistema de Informações sobre Mortalidade (SIM/Datasus).
  - Numerador: óbitos com **causa básica** em acidentes de transporte (CID-10 **V01–V99**),
    no arquivo nacional de causas externas `DOEXT{AA}.dbc`, em
    `dissemin/publicos/SIM/CID10/DOFET/` do FTP do Datasus — o diretório de **dados finais**,
    com um arquivo por ano.
  - Denominador: população residente, série IPEADATA `AVIOL12_VIOLPOP` — a mesma usada em
    `homicidios`, para que as duas taxas sejam comparáveis.
- **Definição:** óbitos por acidentes de transporte por 100 mil habitantes, por local de
  residência, no ano-calendário. Todos os modos (pedestre, ciclista, motocicleta, automóvel,
  caminhão, ônibus, aquaviário e aéreo), em vias urbanas e rodovias.
- **Fórmula:** `taxa = 100000 × óbitos V01–V99 ÷ população residente`, no mesmo ano. A
  população é uma série única em todos os anos; não há denominador fixo nem taxa herdada de
  terceiro.
- **Unidade:** por 100 mil habitantes (`unidade.tipo: numero`, `simbolo: "por 100 mil"`,
  `casas: 2`); `eixo: por-100-mil`, `medida: nivel`, `granularidade: anual`.
- **Período publicado:** `1996-01-01` até o último ano fechado do SIM.
- **Arquivo gerado:** `indicadores/mobilidade/mortes-transito-anual.json`.

## O que este número é — e o que ele não é

- **É** a contagem de mortes cuja causa básica declarada é acidente de transporte, dividida
  pela população residente, × 100 mil. Inclui pedestres, ciclistas, motociclistas, ocupantes
  de automóveis, caminhões e ônibus, e também transporte aquaviário e aéreo.
- **Não é** número de sinistros (um sinistro pode matar várias pessoas), nem de feridos, nem
  de multas, nem de veículos envolvidos. Também não é o recorte de rodovias federais da PRF,
  que cobre só a malha federal e conta por ocorrência.
- **Não é** a morte de trânsito classificada como evento de intenção indeterminada: lesões
  de trânsito codificadas em Y10–Y34 não entram, porque a causa básica não é V01–V99. O
  número é o do bloco CID-10 "acidentes de transporte" do SIM.
- **V01–V99 é o bloco inteiro:** inclui transporte aquaviário (V90–V94), aéreo (V95–V97) e
  os códigos residuais V98–V99. Quem publica apenas transporte **terrestre** (V01–V89)
  chega a um total menor — 2,9% em 2024. A conferência externa abaixo mostra a diferença
  contra o ONSV, que usa V01–V89.

## Por que microdados do `.dbc` e não a taxa pronta de terceiro

O TabNet do Datasus não é uma API estável: as tabelas do SIM testadas em 2026-11-04
devolveram "Tabela de conversao nao encontrada" (ver `MortalidadeInfantil.ipynb`), e o motor
novo só serve as tabelas do IDB. Além disso, a taxa pronta de terceiros troca de denominador
ao longo do tempo — o Atlas da Violência, por exemplo, publica 16,2 por 100 mil para
transporte terrestre em 2023, contra 16,41 do mesmo recorte calculado aqui com uma população
única.

Por isso o notebook **recalcula** a taxa a partir dos microdados do SIM: baixa os arquivos
de causas externas e conta os registros com `CAUSABAS` em V01–V99. O `.dbc` do Datasus é um
DBF com o **corpo comprimido** (PKWare DCL): o notebook baixa o arquivo inteiro para um
diretório temporário, descomprime com `pyreaddbc` e lê o DBF resultante — o cabeçalho diz
quantos registros existem, os descritores de campo dão a posição de `CAUSABAS` e `DTOBITO`,
e cada registro é lido no tamanho fixo declarado. É diferente da mortalidade infantil, que
só precisava do total de registros no cabeçalho: aqui a **causa de cada óbito** importa,
então o arquivo inteiro é necessário.

## Ano fechado

O diretório `DOFET` é o de **dados finais** do SIM: um ano entra quando o Datasus publica o
arquivo final, tipicamente entre 8 e 12 meses depois do fim do ano. O notebook exclui o ano
corrente mesmo que ele apareça no diretório e, antes de publicar, confere que o último ano
tem os **12 meses** de óbito no arquivo — um arquivo parcial não vira ponto da série.
Reexecutar o notebook muda valores só se o Datasus revisar a base; aí é revisão de fonte,
registrada em `docs/metodologia.md`.

A série começa em 1996 porque é o primeiro ano inteiro em CID-10: não há emenda com a CID-9.

## Execução sem sobrescrever o publicado

A variável de ambiente `MOBILIDADE_SAIDA` redireciona o arquivo:

```
MOBILIDADE_SAIDA=/caminho/temporario jupyter nbconvert --execute --to notebook Transito.ipynb
```


In [1]:
# O pyreaddbc não vem instalado no Colab; instala a versão fixada em pyproject.toml
# só quando falta. No kernel local do uv, o import já resolve.
try:
    import pyreaddbc  # noqa: F401
except ImportError:
    import subprocess, sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "pyreaddbc==2.0.4"]
    )


In [2]:
from concurrent.futures import ThreadPoolExecutor
from datetime import date
from pathlib import Path
import ftplib
import json
import math
import os
import re
import struct
import tempfile
import time
from decimal import Decimal, ROUND_HALF_UP

import requests
from pyreaddbc import dbc2dbf

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()
if os.environ.get("DADOS_RAIZ"):
    PASTA = RAIZ / "indicadores" / "mobilidade"
    if not PASTA.is_dir():
        raise SystemExit(f"família não encontrada em {PASTA}")
else:
    PASTA = RAIZ
DESTINO = Path(os.environ.get("MOBILIDADE_SAIDA", PASTA))


def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name


FTP_HOST = "ftp.datasus.gov.br"
# Microdados FINAIS do SIM: um arquivo nacional de causas externas por ano.
DIR_SIM = "/dissemin/publicos/SIM/CID10/DOFET"
URL_POPULACAO = (
    "https://www.ipeadata.gov.br/api/odata4/"
    "ValoresSerie(SERCODIGO='AVIOL12_VIOLPOP')"
)
INICIO = 1996
# Conexões FTP simultâneas. Cada worker abre a própria conexão.
CONEXOES = 6

## Coleta: quais anos entram

O diretório do FTP é listado a cada execução — o notebook não lê o JSON publicado para
decidir o que buscar. O nome do arquivo segue `DOEXT{AA}.dbc`, com dois dígitos no ano
(pivô em 90: `DOEXT96` é 1996 e `DOEXT25` é 2025). O ano corrente é descartado mesmo que
apareça no diretório: a base do ano em curso está incompleta por construção.

In [3]:
def listar(caminho):
    """Lista um diretório do FTP Datasus, com reconexão e espera crescente."""
    for tentativa in range(1, 6):
        try:
            ftp = ftplib.FTP(FTP_HOST, timeout=60)
            ftp.login()
            itens = []
            ftp.retrlines("NLST " + caminho, itens.append)
            ftp.quit()
            return [Path(item).name for item in itens]
        except Exception:
            if tentativa == 5:
                raise
            time.sleep(2 ** tentativa)


def ano_do_doext(nome):
    m = re.fullmatch(r"DOEXT(\d{2})\.dbc", nome, re.IGNORECASE)
    if not m:
        return None
    aa = int(m.group(1))
    # Pivô de dois dígitos: 96–99 → 1996–1999 e 00–89 → 2000–2089. O corte em 90 (e
    # não em 25) não depende do ano corrente nem transforma DOEXT26 em 1926.
    return 2000 + aa if aa < 90 else 1900 + aa


arquivos = {}
for nome in listar(DIR_SIM):
    ano = ano_do_doext(nome)
    if ano is not None and ano >= INICIO:
        arquivos[ano] = f"{DIR_SIM}/{nome}"

ano_corrente = date.today().year
excluidos = sorted(ano for ano in arquivos if ano >= ano_corrente)
for ano in excluidos:
    del arquivos[ano]
anos = sorted(arquivos)

if not anos:
    raise ValueError("nenhum arquivo DOEXT encontrado no FTP do Datasus")

print(f"anos com arquivo final do SIM: {anos[0]}–{anos[-1]} ({len(anos)} anos)")
print(f"ano corrente ({ano_corrente}) excluído: {excluidos}")

anos com arquivo final do SIM: 1996–2025 (30 anos)
ano corrente (2026) excluído: []


## Leitura do `.dbc`: baixar, descomprimir e contar

Cada ano é processado num worker próprio, com uma conexão FTP curta e um diretório
temporário que é apagado no fim. O `pyreaddbc.dbc2dbf` descomprime o corpo do `.dbc` para um
DBF comum; a contagem então percorre os registros de tamanho fixo e lê dois campos:

- `CAUSABAS` — CID-10 da causa básica. Entra na série o registro cujo **grupo** é `V` com
  dois dígitos entre `01` e `99`. Códigos de três caracteres (`V99`, por exemplo, sem o
  quarto dígito) também entram: são a mesma categoria.
- `DTOBITO` — data do óbito, usada só para conferir se o último ano tem os 12 meses.

Os registros marcados como apagados (marcador `*` no primeiro byte) não são contados. O
arquivo nacional reúne as declarações por município de residência; o notebook conta todos os
registros, sem filtro de UF nem de local de ocorrência.

In [4]:
def baixar(caminho, destino):
    """Baixa um arquivo do FTP Datasus, com reconexão e espera crescente."""
    for tentativa in range(1, 6):
        try:
            ftp = ftplib.FTP(FTP_HOST, timeout=180)
            ftp.login()
            with destino.open("wb") as saida:
                ftp.retrbinary("RETR " + caminho, saida.write, blocksize=65536)
            ftp.quit()
            return
        except Exception:
            if tentativa == 5:
                raise
            time.sleep(2 ** tentativa)


def contar_por_causa(dbf):
    """Conta os óbitos por acidentes de transporte num DBF descomprimido do SIM.

    O cabeçalho DBF (bytes 4–7) diz quantos registros existem e os bytes 10–11 dizem o
    tamanho de cada registro; os descritores de campo (32 bytes cada, após o cabeçalho de
    32) dão a posição de ``CAUSABAS`` e ``DTOBITO``. Devolve
    ``(registros, V01–V99, V01–V89, meses_com_óbito)``.
    """
    with dbf.open("rb") as arquivo:
        cabecalho = arquivo.read(32)
        registros, tamanho_cabecalho, tamanho_registro = struct.unpack(
            "<IHH", cabecalho[4:12]
        )
        campos = []
        for _ in range((tamanho_cabecalho - 33) // 32):
            descritor = arquivo.read(32)
            nome = descritor[:11].split(b"\x00")[0].decode("ascii")
            campos.append((nome, descritor[16]))
        arquivo.seek(tamanho_cabecalho)
        corpo = arquivo.read(registros * tamanho_registro)

    if len(corpo) != registros * tamanho_registro:
        raise ValueError("DBF truncado: o corpo não tem os registros do cabeçalho")

    deslocamento = 1  # o primeiro byte de cada registro é a marca de exclusão
    posicao = {}
    for nome, tamanho in campos:
        posicao[nome] = (deslocamento, tamanho)
        deslocamento += tamanho
    if deslocamento != tamanho_registro:
        raise ValueError(
            f"descritores somam {deslocamento} bytes, registro tem {tamanho_registro}"
        )

    inicio_causa, tamanho_causa = posicao["CAUSABAS"]
    inicio_data, tamanho_data = posicao["DTOBITO"]

    total = 0
    acidentes = 0
    terrestres = 0
    meses = set()
    for i in range(registros):
        registro = corpo[i * tamanho_registro:(i + 1) * tamanho_registro]
        if registro[0:1] == b"*":
            continue  # registro marcado como apagado
        total += 1
        causa = (
            registro[inicio_causa:inicio_causa + tamanho_causa]
            .decode("latin-1")
            .strip()
        )
        grupo = (
            int(causa[1:3])
            if len(causa) >= 3 and causa[0] == "V" and causa[1:3].isdigit()
            else 0
        )
        if 1 <= grupo <= 99:
            acidentes += 1
            if grupo <= 89:
                terrestres += 1
        mes = (
            registro[inicio_data:inicio_data + tamanho_data]
            .decode("latin-1")
            .strip()[2:4]
        )
        if mes.isdigit():
            meses.add(int(mes))

    return total, acidentes, terrestres, meses


def contar_ano(ano):
    """Baixa o .dbc do ano, descomprime e conta (worker: uma conexão FTP própria)."""
    with tempfile.TemporaryDirectory() as temporario:
        dbc = Path(temporario) / "DOEXT.dbc"
        dbf = Path(temporario) / "DOEXT.dbf"
        baixar(arquivos[ano], dbc)
        dbc2dbf(str(dbc), str(dbf))
        total, acidentes, terrestres, meses = contar_por_causa(dbf)
    return ano, total, acidentes, terrestres, meses


totais = {}
acidentes = {}
terrestres = {}
meses = {}
inicio = time.perf_counter()
# O ``map`` devolve na ordem dos anos, então a saída não depende de quem termina primeiro.
with ThreadPoolExecutor(max_workers=CONEXOES) as executor:
    for ano, total, por_acidente, por_terrestre, meses_do_ano in executor.map(
        contar_ano, anos
    ):
        totais[ano] = total
        acidentes[ano] = por_acidente
        terrestres[ano] = por_terrestre
        meses[ano] = meses_do_ano
        print(
            f"{ano}: {total:>7,} óbitos por causas externas, "
            f"{por_acidente:>6,} por acidentes de transporte"
        )

# Um ano só é publicado se o arquivo tiver os 12 meses do ano: um arquivo parcial não
# vira ponto da série. A checagem vale para o último ano; os anos antigos têm formatos
# de data legados (1996 tem DTOBITO em branco ou com dois dígitos).
ultimo = anos[-1]
if meses[ultimo] != set(range(1, 13)):
    raise ValueError(
        f"{ultimo} não tem os 12 meses de óbito no SIM "
        f"(meses: {sorted(meses[ultimo])}); o ano não está fechado"
    )

print()
print(f"coleta: {time.perf_counter() - inicio:.1f} s em {len(anos)} arquivos")
print(f"conferência: {ultimo} tem os 12 meses do ano")

1996: 119,156 óbitos por causas externas, 35,545 por acidentes de transporte
1997: 119,550 óbitos por causas externas, 35,756 por acidentes de transporte
1998: 117,690 óbitos por causas externas, 31,026 por acidentes de transporte
1999: 116,894 óbitos por causas externas, 30,118 por acidentes de transporte
2000: 118,397 óbitos por causas externas, 29,645 por acidentes de transporte
2001: 120,954 óbitos por causas externas, 31,031 por acidentes de transporte


2002: 126,550 óbitos por causas externas, 33,288 por acidentes de transporte
2003: 126,657 óbitos por causas externas, 33,620 por acidentes de transporte


2004: 127,470 óbitos por causas externas, 35,674 por acidentes de transporte


2005: 127,633 óbitos por causas externas, 36,611 por acidentes de transporte


2006: 128,388 óbitos por causas externas, 37,249 por acidentes de transporte
2007: 131,032 óbitos por causas externas, 38,419 por acidentes de transporte


2008: 135,936 óbitos por causas externas, 39,211 por acidentes de transporte
2009: 138,697 óbitos por causas externas, 38,469 por acidentes de transporte


... (3 outputs left out)

2016: 155,861 óbitos por causas externas, 38,265 por acidentes de transporte


2017: 158,657 óbitos por causas externas, 36,430 por acidentes de transporte
2018: 150,814 óbitos por causas externas, 33,625 por acidentes de transporte
2019: 142,800 óbitos por causas externas, 32,879 por acidentes de transporte
2020: 146,038 óbitos por causas externas, 33,716 por acidentes de transporte


2021: 149,322 óbitos por causas externas, 35,032 por acidentes de transporte
2022: 152,945 óbitos por causas externas, 34,892 por acidentes de transporte


2023: 154,199 óbitos por causas externas, 35,938 por acidentes de transporte
2024: 159,534 óbitos por causas externas, 38,253 por acidentes de transporte
2025: 156,883 óbitos por causas externas, 37,954 por acidentes de transporte

coleta: 45.9 s em 30 arquivos
conferência: 2025 tem os 12 meses do ano


## Coleta do denominador

A população é a série `AVIOL12_VIOLPOP` do Ipeadata: "População residente —
Estimativa/TCU (ajustada)", a mesma usada em `homicidios`. A metadata declara que ela é o
ponderador das taxas de segurança pública do Atlas da Violência, que os ajustes judiciais
não são considerados e que **2022 e 2023 foram interpolados exponencialmente entre as
estimativas de 2021 e 2024**.

O endpoint OData não aceita filtro por território; a série completa (todas as unidades) é
baixada e filtrada para `NIVNOME = "Brasil"` / `TERCODIGO = "0"`. É feito uma vez por
execução, sem cache e sem leitura do JSON publicado. A limitação de 2022–2023 é do
denominador, não do numerador.

In [5]:
def baixar_json(url, tentativas=4, timeout=600):
    """Baixa JSON com espera crescente; a API do Ipeadata oscila."""
    for tentativa in range(tentativas):
        try:
            resposta = requests.get(url, timeout=timeout)
            resposta.raise_for_status()
            return resposta.json()
        except (requests.RequestException, ValueError):
            if tentativa == tentativas - 1:
                raise
            time.sleep(2 ** tentativa)


bruto_populacao = baixar_json(URL_POPULACAO)
populacao = {}
for item in bruto_populacao["value"]:
    if item.get("NIVNOME") == "Brasil" and item.get("TERCODIGO") == "0":
        ano = int(item["VALDATA"][:4])
        populacao[ano] = float(item["VALVALOR"])

if not populacao:
    raise ValueError("a série AVIOL12_VIOLPOP não retornou o Brasil")

print(f"população: {len(populacao)} anos ({min(populacao)}–{max(populacao)})")
for ano in (1996, 2000, 2012, 2023, 2024, 2025):
    print(f"  {ano}: {populacao[ano]:,.0f} habitantes")

população: 46 anos (1980–2025)
  1996: 157,070,163 habitantes
  2000: 169,799,170 habitantes
  2012: 199,239,473 habitantes
  2023: 212,572,795 habitantes
  2024: 212,577,978 habitantes
  2025: 213,415,160 habitantes


## Cálculo da taxa

Cada ponto é `100000 × óbitos V01–V99 ÷ população` do **mesmo ano**. Não há média móvel,
não há taxa herdada: a contagem e a população são do ano-calendário. O valor é arredondado
para 2 casas (meio para cima) na gravação; `unidade.casas` é só exibição.

A conferência externa compara o recorte de **transporte terrestre** (V01–V89, que é o do
ONSV e o do Atlas da Violência) com as contagens que o ONSV publica para 2019–2024 no
relatório "Dados Consolidados de Óbitos no Trânsito Brasileiro – 2024"
(`https://onsv.github.io/analise-datasus-2024/datasus2024.html`) — ela não entra na série,
que é V01–V99.

In [6]:
def arredondar(valor):
    """Arredonda para 2 casas com meio para cima (convenção comercial)."""
    return float(Decimal(repr(valor)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))


comuns = [ano for ano in anos if ano in populacao]
if comuns != list(range(INICIO, comuns[-1] + 1)):
    raise ValueError(f"lacuna entre os anos com óbitos e população: {comuns}")

# Fórmula: taxa = 100000 × óbitos por acidentes de transporte ÷ população residente.
taxa = {ano: arredondar(100000 * acidentes[ano] / populacao[ano]) for ano in comuns}

print(f"{'ano':>4} {'óbitos':>8} {'população':>13} {'taxa':>6}")
for ano in comuns:
    print(f"{ano:>4} {acidentes[ano]:>8,} {populacao[ano]:>13,.0f} {taxa[ano]:>6.2f}")

# Contagens do ONSV ("Dados Consolidados de Óbitos no Trânsito Brasileiro – 2024") para
# transporte terrestre, V01–V89. A comparação é só de conferência: a série publicada é
# V01–V99 e o denominador do ONSV não é o desta série.
ONSV_TERRESTRE = {
    2019: 31945,
    2020: 32716,
    2021: 33813,
    2022: 33894,
    2023: 34881,
    2024: 37150,
}
print()
print("Conferência externa (não entra na série): transporte terrestre V01–V89")
for ano, publicado in sorted(ONSV_TERRESTRE.items()):
    print(
        f"  {ano}: {terrestres[ano]:,} no SIM contra {publicado:,} publicados "
        f"({terrestres[ano] - publicado:+,})"
    )

 ano   óbitos     população   taxa
1996   35,545   157,070,163  22.63
1997   35,756   159,636,297  22.40
1998   31,026   161,790,182  19.18
1999   30,118   163,947,436  18.37
2000   29,645   169,799,170  17.46
2001   31,031   172,383,140  18.00
2002   33,288   174,630,259  19.06
2003   33,620   180,619,076  18.61
2004   35,674   182,908,586  19.50
2005   36,611   185,150,821  19.77
2006   37,249   187,335,127  19.88
2007   38,419   189,462,739  20.28
2008   39,211   191,532,851  20.47
2009   38,469   193,544,469  19.88
2010   43,908   195,494,818  22.46
2011   44,553   197,394,037  22.57
2012   46,051   199,239,473  23.11
2013   43,452   201,029,717  21.61
2014   44,823   202,765,334  22.11
2015   39,543   204,447,367  19.34
2016   38,265   206,098,835  18.57
2017   36,430   206,789,329  17.62
2018   33,625   208,479,286  16.13
2019   32,879   210,131,314  15.65
2020   33,716   211,739,687  15.92
2021   35,032   213,317,639  16.42
2022   34,892   212,820,386  16.40
2023   35,938   212,

## Escrita do JSON do contrato

O arquivo é gravado inteiro, com todas as chaves da raiz e sem campo de apresentação. As
datas seguem a convenção do contrato: **dia 01 do início do período de referência**, um
ponto por ano (C11). O `fonte` nomeia o SIM/Datasus e o recorte CID-10, e declara que a taxa
é calculada pelo site. Não há `urlFonte`: o recorte é um caminho `ftp://` e C7.1 só aceita
`https://` — os caminhos exatos estão no markdown e em `docs/metodologia.md`.

In [7]:
def pontos(serie):
    return [
        {"data": f"{ano}-01-01", "valor": valor}
        for ano, valor in sorted(serie.items())
    ]


FONTE = (
    "Ministério da Saúde — SIM/Datasus (CID-10 V01–V99); "
    "taxa por 100 mil calculada pelo site"
)

serie = {
    "versao": 1,
    "id": "mortes-transito-anual",
    "unidade": {"tipo": "numero", "simbolo": "por 100 mil", "casas": 2},
    "eixo": "por-100-mil",
    "medida": "nivel",
    "granularidade": "anual",
    "fonte": FONTE,
    "dados": pontos(taxa),
}

DESTINO.mkdir(parents=True, exist_ok=True)
arquivo = DESTINO / "mortes-transito-anual.json"
with arquivo.open("w", encoding="utf-8") as saida:
    json.dump(serie, saida, ensure_ascii=False, indent=2)
    saida.write("\n")
print(f"{len(serie['dados'])} pontos escritos em {relativo(arquivo)}")

30 pontos escritos em indicadores/mobilidade/mortes-transito-anual.json


## Validação inline

A célula final falha se o contrato, a ordem, a unicidade, a continuidade dos anos, o dia 01,
a finitude dos valores, a fórmula ou a gravação em disco forem violados. Também confere os
números citados em `docs/metodologia.md` e usa pisos de cobertura para o caso de o SIM
fechar mais um ano.

As referências de valor são âncoras: se o Datasus revisar a base, elas falham de propósito,
para que a revisão seja conferida e registrada antes de o número mudar.

In [8]:
CHAVES_RAIZ = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "dados",
}

assert set(serie) == CHAVES_RAIZ, sorted(set(serie) ^ CHAVES_RAIZ)
assert serie["versao"] == 1
assert serie["id"] == "mortes-transito-anual"
assert serie["unidade"] == {"tipo": "numero", "simbolo": "por 100 mil", "casas": 2}
assert serie["eixo"] == "por-100-mil"
assert serie["medida"] == "nivel"
assert serie["granularidade"] == "anual"
assert isinstance(serie["fonte"], str) and serie["fonte"]
# C7.1: urlFonte é opcional e, quando existe, precisa ser https://. O recorte desta série
# é um caminho FTP (ftp:// é proibido), então não há urlFonte — os caminhos exatos ficam
# no markdown e em docs/metodologia.md.
assert "urlFonte" not in serie, "urlFonte só entra com https://; o recorte aqui é ftp://"

dados = serie["dados"]
assert dados, "série vazia"
datas = [ponto["data"] for ponto in dados]
anos_lista = [int(data[:4]) for data in datas]

# Série anual: todo ponto em AAAA-01-01 (C15). A paridade bienal do C6 vale para
# granularidade "bienal", não para esta série.
assert datas[0] == "1996-01-01", datas[0]
assert all(data.endswith("-01-01") for data in datas), "dia diferente de 01"
assert all(a < b for a, b in zip(datas, datas[1:])), "datas fora de ordem"
assert anos_lista == list(range(anos_lista[0], anos_lista[-1] + 1)), (
    "ano faltando ou repetido"
)

for ponto in dados:
    assert set(ponto) == {"data", "valor"}, sorted(ponto)
    assert not isinstance(ponto["valor"], bool), ponto
    assert isinstance(ponto["valor"], (int, float)), ponto
    assert math.isfinite(ponto["valor"]), ponto

# Fórmula: cada ponto é 100000 × óbitos V01–V99 ÷ população do mesmo ano.
for ponto in dados:
    ano = int(ponto["data"][:4])
    assert ponto["valor"] == arredondar(100000 * acidentes[ano] / populacao[ano]), ponto

# Confere a gravação em disco.
with (DESTINO / "mortes-transito-anual.json").open(encoding="utf-8") as arquivo:
    assert json.load(arquivo) == serie, "arquivo gravado difere do objeto"


def valor(ano):
    return {int(p["data"][:4]): p["valor"] for p in dados}[ano]


# Cobertura: o começo é fixo; o fim é o último ano fechado do SIM, então só pode crescer.
assert int(dados[-1]["data"][:4]) >= 2025, (
    f"último ano {dados[-1]['data']}: a série publicada ia até 2025; "
    "se perdeu um ano, veja docs/metodologia.md"
)
assert len(dados) >= 30, (
    f"{len(dados)} pontos: a série publicada tinha 30 (1996–2025); se a fonte fechou "
    "mais um ano, atualize o piso; se perdeu um ano, veja docs/metodologia.md"
)

# Números citados na metodologia e na evidência. Se falharem, é revisão de fonte: confira
# os microdados e registre em docs/metodologia.md antes de mexer no valor.
assert valor(1996) == 22.63, "1996 22.63 — revisão de fonte? veja docs/metodologia.md"
assert valor(2000) == 17.46, "2000 17.46 — revisão de fonte? veja docs/metodologia.md"
assert valor(2012) == 23.11, "2012 23.11 — revisão de fonte? veja docs/metodologia.md"
assert valor(2023) == 16.91, "2023 16.91 — revisão de fonte? veja docs/metodologia.md"
assert valor(2024) == 17.99, "2024 17.99 — revisão de fonte? veja docs/metodologia.md"
assert valor(2025) == 17.78, "2025 17.78 — revisão de fonte? veja docs/metodologia.md"

print(
    f"ok: {len(dados)} pontos ({datas[0]} a {datas[-1]}); fórmula, dia 01, ordem, "
    "cobertura e contrato conferidos."
)

ok: 30 pontos (1996-01-01 a 2025-01-01); fórmula, dia 01, ordem, cobertura e contrato conferidos.
